# **Executive Summary**



This project develops a machine learning-based system to predict emergency department (ED) wait times using a synthetic dataset representing patient and hospital conditions. The primary objective is to provide accurate wait time estimates while maintaining interpretability through explainable AI techniques.

An XGBoost regression model was selected as the main predictive model due to its strong performance on structured data. The model achieved a Mean Absolute Error (MAE) of approximately 12 minutes, meeting the predefined success criterion of ≤15 minutes. Baseline models, including Linear Regression and Random Forest, were used for comparison to validate model selection.

To enhance transparency, SHAP (SHapley Additive exPlanations) was used to provide both global and local explanations, allowing key drivers of predictions to be identified. Results show that factors such as urgency level, time of visit, and seasonal effects significantly influence predicted wait times.

To demonstrate practical application, two interactive dashboards were developed. An in-notebook dashboard using ipywidgets allows for quick experimentation within the analysis environment, while a Streamlit-based web application provides a more polished, user-friendly interface for real-time predictions and explanations.

While effective as a prototype, the system assumes access to internal hospital data (e.g., staffing levels), which would typically be automatically retrieved in a real-world deployment.

Overall, the project demonstrates the feasibility of combining predictive modelling with explainable AI to support decision-making in healthcare environments.

# **Exploratory Data Analysis**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# display settings
pd.set_option('display.max_columns', None)

# load dataset
df = pd.read_csv('ER Wait Time Dataset.csv')

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Total Wait Time (min)'], bins=50, kde=True)
plt.title('Distribution of Total Wait Time')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Urgency Level', y='Total Wait Time (min)', data=df)
plt.title('Wait Time by Urgency Level')
plt.show()

In [ ]:
sns.scatterplot(x='Nurse-to-Patient Ratio', y='Total Wait Time (min)', data=df)
plt.title('Staff Ratio vs Wait Time')
plt.show()

In [ ]:
sns.boxplot(x='Specialist Availability', y='Total Wait Time (min)', data=df)
plt.show()

In [ ]:
sns.boxplot(x='Time of Day', y='Total Wait Time (min)', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.boxplot(x='Day of Week', y='Total Wait Time (min)', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.boxplot(x='Season', y='Total Wait Time (min)', data=df)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Region', y='Total Wait Time (min)', data=df)
plt.title('Wait Time by Region (Urban vs Rural)')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## **Feature Analysis**

### **Key Insights from Exploratory Data Analysis**

- The distribution of total wait time is right-skewed, indicating that most patients experience shorter waiting times, while a smaller proportion experience significantly longer delays. This suggests the presence of extreme cases.

- A strong relationship is observed between urgency level and waiting time. Patients classified as critical or high urgency experience significantly shorter wait times compared to medium and low urgency cases, reflecting prioritisation in emergency care systems.

- The nurse-to-patient ratio shows a positive relationship with waiting time, where higher values are associated with longer delays. This suggests that the ratio is encoded as patients per nurse rather than nurses per patient. This interpretation is important, as it indicates that higher workload per nurse contributes to increased waiting times, and highlights a potential limitation in how the variable is labelled.

- Time-based patterns indicate that waiting times tend to be higher during evening periods compared to early morning, suggesting increased demand during peak hours.

- Analysis across days of the week shows some variation in waiting times, with certain days (e.g. Monday and Friday) exhibiting slightly higher delays, potentially due to increased patient inflow.

- Seasonal variation is present but not strongly pronounced, suggesting that seasonal effects may have a limited impact compared to operational factors such as staffing and urgency.

- Regional analysis shows that urban and rural hospitals exhibit similar distributions of waiting times, indicating that region alone does not significantly influence waiting time in this dataset.

- Extremely high correlations are observed between stage-level waiting times (registration, triage, and consultation) and total wait time. These variables directly contribute to the target and would introduce data leakage if used in predictive modelling.

# **Predictive Modelling**

## Preprocessing

In [ ]:
# Create a copy for modelling
df_model = df.copy()

# Columns to drop
drop_cols = [
    'Visit ID',
    'Patient ID',
    'Hospital ID',
    'Hospital Name',
    'Time to Registration (min)',
    'Time to Triage (min)',
    'Time to Medical Professional (min)',
    'Patient Outcome',
    'Patient Satisfaction'
]

df_model = df_model.drop(columns=drop_cols)

df_model.head()

In [ ]:
df_model.columns

In [ ]:
df_model['Visit Date'] = pd.to_datetime(df_model['Visit Date'], dayfirst=True)

In [ ]:
df_model['Visit Month'] = df_model['Visit Date'].dt.month
df_model['Visit Hour'] = df_model['Visit Date'].dt.hour
df_model['Visit Day'] = df_model['Visit Date'].dt.day

In [ ]:
df_model = df_model.drop(columns=['Visit Date'])

In [ ]:
target = 'Total Wait Time (min)'

X = df_model.drop(columns=[target])
y = df_model[target]

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_encoded = X_encoded.astype(int)

X_encoded.head()

In [ ]:
print("Original X shape:", X.shape)
print("Encoded X shape:", X_encoded.shape)
print("Target shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

### **Preprocessing Summary**

- Identifier columns such as Visit ID, Patient ID, Hospital ID, and Hospital Name were removed as they do not provide predictive value.

- Stage-level waiting time variables (registration, triage, and consultation times) were excluded to prevent data leakage, as they are direct components of the target variable.

- Post-visit variables including Patient Outcome and Patient Satisfaction were also removed, as they are not available at prediction time and would introduce leakage.

- The Visit Date column was converted into datetime format, and additional temporal features including visit month, visit hour, and visit day were extracted.

- Categorical variables were transformed using one-hot encoding to ensure compatibility with machine learning models.

- Boolean features generated during encoding were converted into integer format for consistency across models.

- The dataset was split into training and testing sets using an 80/20 split to evaluate model performance.

- A random train-test split was used rather than a time-based split. While a temporal split may better reflect real-world deployment, the random split was chosen to ensure balanced representation across training and test sets.

## Main Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Initialize model
lr = LinearRegression()

# Train
lr.fit(X_train, y_train)

# Predict
y_pred_lr = lr.predict(X_test)

# Evaluate
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print("Linear Regression MAE:", mae_lr)
print("Linear Regression RMSE:", rmse_lr)

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBRegressor

# Initialize model
xgb = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

# Train
xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Evaluate
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print("XGBoost MAE:", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)

## **Experiment**

### Experimental Settings

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize model
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Train
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("Random Forest MAE:", mae_rf)
print("Random Forest RMSE:", rmse_rf)

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error

mape_lr = mean_absolute_percentage_error(y_test, y_pred_lr) * 100
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb) * 100
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf) * 100

print("Linear Regression MAPE:", mape_lr)
print("XGBoost MAPE:", mape_xgb)
print("Random Forest MAPE:", mape_rf)

In [ ]:
print("\nModel Comparison:")
print(f"Linear Regression -> MAE: {mae_lr:.2f}, RMSE: {rmse_lr:.2f}, MAPE: {mape_lr:.2f}")
print(f"Random Forest     -> MAE: {mae_rf:.2f}, RMSE: {rmse_rf:.2f}, MAPE: {mape_rf:.2f}")
print(f"XGBoost           -> MAE: {mae_xgb:.2f}, RMSE: {rmse_xgb:.2f}, MAPE: {mape_xgb:.2f}")

### Experiment Summary

In addition to the primary model, a Random Forest regressor was implemented to provide an intermediate comparison between simple and more complex modelling approaches.

All models were trained on the same training dataset and evaluated on a held-out test set using consistent performance metrics (MAE, RMSE, and MAPE), ensuring a fair comparison.

Default hyperparameters were used across all models to establish a baseline before any further optimisation. This allows performance differences to be attributed to model characteristics rather than tuning.

# **Evaluation**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

metrics = {
    'MAE': [mae_lr, mae_rf, mae_xgb],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb],
    'MAPE (%)': [mape_lr, mape_rf, mape_xgb]
}

models = ['Linear Regression', 'Random Forest', 'XGBoost']

for i, (metric, values) in enumerate(metrics.items()):
    axs[i].bar(models, values)
    axs[i].set_title(metric)
    axs[i].set_ylabel('Score')
    axs[i].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Evaluation Summary

The model was evaluated using standard regression metrics, including Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Mean Absolute Percentage Error (MAPE), to assess predictive accuracy and reliability.

Linear Regression achieved an MAE of 17.91 minutes, RMSE of 24.75 minutes, and MAPE of 46.65%, indicating relatively poor performance.

Random Forest significantly improved performance, achieving an MAE of 12.97 minutes, RMSE of 18.93 minutes, and MAPE of 19.73%.

XGBoost achieved the best performance across all metrics, with an MAE of 12.07 minutes, RMSE of 17.58 minutes, and MAPE of 18.51%.

These results demonstrate that tree-based models outperform linear models for this task, suggesting that the relationship between input features and wait time is non-linear.

Importantly, the XGBoost model meets the predefined success criterion of MAE ≤ 15 minutes, demonstrating that the model performs at a practically useful level.

While a random train-test split was used, a time-based split may better reflect real-world deployment scenarios. Future work could incorporate temporal validation to provide a more robust assessment of model generalisation.

# **Testing and Validation**

In [ ]:
# -----------------------------
# Scenario-Based Testing
# -----------------------------

test_cases = [
    {
        "name": "Low urgency, busy conditions",
        "Urgency Level": "Low",
        "Day of Week": "Monday",
        "Season": "Winter"
    },
    {
        "name": "High urgency case",
        "Urgency Level": "Critical",
        "Day of Week": "Sunday",
        "Season": "Summer"
    },
    {
        "name": "Moderate conditions",
        "Urgency Level": "Medium",
        "Day of Week": "Wednesday",
        "Season": "Spring"
    }
]

for case in test_cases:
    input_dict = {col: 0 for col in X_encoded.columns}

    # Example minimal encoding (adjust if needed)
    for col in X_encoded.columns:
        if case["Urgency Level"] in col:
            input_dict[col] = 1
        if case["Day of Week"] in col:
            input_dict[col] = 1
        if case["Season"] in col:
            input_dict[col] = 1

    input_df = pd.DataFrame([input_dict])
    pred = xgb.predict(input_df)[0]

    print(f"{case['name']} → Predicted Wait: {pred:.1f} minutes")

### Scenario-Based Testing

A set of test scenarios was used to assess whether model predictions aligned with expected real-world behaviour.

- A low urgency case during busy conditions (Monday, Winter) resulted in a predicted wait time of approximately 277 minutes, reflecting the prioritisation of more critical patients and increased demand.
- A high urgency (critical) case produced a significantly shorter predicted wait time of around 20 minutes, demonstrating that the model correctly captures prioritisation in emergency care.
- A moderate scenario resulted in an intermediate prediction of approximately 68 minutes, suggesting the model captures gradual variation in waiting times.

These results indicate that the model behaves consistently with real-world expectations. However, some less intuitive outputs observed during interactive testing highlight the limitations of the synthetic dataset and the representation of categorical variables. This reinforces the importance of validating model behaviour beyond aggregate performance metrics.

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(y_test, y_pred_xgb, alpha=0.3)

# perfect prediction line
max_val = max(y_test.max(), y_pred_xgb.max())
plt.plot([0, max_val], [0, max_val], linestyle='--')

plt.xlabel('Actual Wait Time (min)')
plt.ylabel('Predicted Wait Time (min)')
plt.title('XGBoost: Predicted vs Actual')

plt.tight_layout()
plt.show()

The predicted vs actual plot shows that the XGBoost model performs well for most observations, with predictions closely following the diagonal line. However, greater deviation is observed at higher wait times, indicating that extreme cases are more difficult to predict accurately.

# **Explainability**

## **SHAP Analysis**

In [ ]:
!pip install shap
import shap

explainer = shap.Explainer(xgb, X_train)
shap_values = explainer(X_test)



In [ ]:
shap.summary_plot(shap_values, X_test)

In [ ]:
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
shap.plots.waterfall(shap_values[0])

### **Explainability Summary**

SHAP analysis shows that urgency level is the most influential predictor of wait time, with low urgency cases significantly increasing predicted wait times. This aligns with real-world emergency department prioritisation.

Temporal factors such as winter season, evening periods, and Mondays also contribute to longer waits, reflecting peak demand patterns.

The nurse-to-patient ratio has a moderate impact, confirming that staffing levels influence wait times, although urgency remains the dominant factor.

Specialist availability shows minimal importance, consistent with earlier EDA findings.

The waterfall plot illustrates how individual features combine to produce a prediction, highlighting the contribution of key factors to a specific case. For example, one instance was predicted to have a wait time of 236.8 minutes, primarily driven by low urgency (+104.71), winter season (+32.34), and Monday attendance (+31.68).


# **Critical Review**



### Strengths

* XGBoost achieved an MAE of 12.07 minutes, meeting the predefined success criterion of ≤15 minutes and demonstrating strong practical predictive utility.
* SHAP analysis provided transparent and interpretable explanations at both global and individual levels, addressing the explainability gap identified in the literature.
* The three-model comparison (Linear Regression, Random Forest, XGBoost) establishes a clear progression from baseline to advanced model, strengthening the justification for model selection.
* The dataset contained no missing values, simplifying preprocessing and reducing the risk of imputation bias.

### Limitations

* The dataset is synthetic, limiting the generalisability of findings to real-world emergency department settings, where patterns are likely more complex.
* A random train-test split was used instead of a time-based split, which may lead to optimistic performance estimates compared to real deployment scenarios.
* The nurse-to-patient ratio appears to be encoded as patients per nurse, which is counterintuitive and introduces ambiguity in interpretation.
* The model shows reduced accuracy for extreme wait time cases, as observed in the predicted vs actual plot, indicating difficulty in modelling rare or high-demand scenarios.
* Specialist availability exhibited minimal predictive importance, suggesting limited variability or weak representation in the dataset.
* The system requires input variables such as staffing levels, bed capacity, and specialist availability, which would not typically be known by patients. As a result, the current prototype is more suited to healthcare professionals or hospital administrators rather than the general public.

### Future Work

* Incorporating real NHS A&E data would significantly improve the external validity and practical applicability of the model.
* Implementing a time-based validation strategy would provide a more realistic evaluation of model performance.
* Hyperparameter tuning (e.g., grid search or Bayesian optimisation) could further enhance XGBoost performance.
* Modelling individual stages of the patient journey (e.g., registration, triage, consultation) could enable more granular and actionable predictions, subject to data availability.
* Integrating the system with NHS data infrastructure would allow real-time retrieval of operational variables such as staffing levels and bed availability. This would remove the need for manual input, significantly improving usability and enabling the system to function as a patient-facing application.

# **Dashboard Implementation**

In [ ]:
import joblib

joblib.dump(xgb, 'xgb_model.pkl')
joblib.dump(X_encoded.columns.tolist(), 'model_columns.pkl')

## **Dashboard (Interactive)**

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import shap

# If you saved these earlier:
# model = joblib.load('xgb_model.pkl')
# model_columns = joblib.load('model_columns.pkl')

# Dropdown options
urgency_options = ['Low', 'Medium', 'High', 'Critical']
region_options = ['Urban', 'Rural']
season_options = ['Winter', 'Spring', 'Summer', 'Fall']
time_options = ['Early Morning', 'Late Morning', 'Afternoon', 'Evening', 'Night']
day_options = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Widgets
urgency = widgets.Dropdown(options=urgency_options, description='Urgency:')
region = widgets.Dropdown(options=region_options, description='Region:')
season = widgets.Dropdown(options=season_options, description='Season:')
time_of_day = widgets.Dropdown(options=time_options, description='Time:')
day_of_week = widgets.Dropdown(options=day_options, description='Day:')
nurse_ratio = widgets.IntSlider(min=1, max=5, value=3, description='Nurse Ratio:')
specialist = widgets.IntSlider(min=0, max=10, value=3, description='Specialists:')
facility = widgets.IntSlider(min=10, max=200, value=80, description='Beds:')
visit_hour = widgets.IntSlider(min=0, max=23, value=12, description='Visit Hour:')
visit_month = widgets.IntSlider(min=1, max=12, value=6, description='Month:')
visit_day = widgets.IntSlider(min=1, max=31, value=15, description='Visit Day:')

predict_btn = widgets.Button(description='Predict Wait Time', button_style='primary')
output = widgets.Output()

# Use model_columns if available, otherwise X_encoded.columns
feature_columns = model_columns if 'model_columns' in globals() else X_encoded.columns.tolist()

def predict(b):
    output.clear_output()

    # Build input row
    input_dict = {col: 0 for col in feature_columns}

    # Numerical
    input_dict['Nurse-to-Patient Ratio'] = nurse_ratio.value
    input_dict['Specialist Availability'] = specialist.value
    input_dict['Facility Size (Beds)'] = facility.value
    input_dict['Visit Month'] = visit_month.value
    input_dict['Visit Hour'] = visit_hour.value
    input_dict['Visit Day'] = visit_day.value

    # Categorical (one-hot, only if column exists)
    for col in [
        f'Region_{region.value}',
        f'Day of Week_{day_of_week.value}',
        f'Season_{season.value}',
        f'Time of Day_{time_of_day.value}',
        f'Urgency Level_{urgency.value}'
    ]:
        if col in input_dict:
            input_dict[col] = 1

    input_df = pd.DataFrame([input_dict])[feature_columns]

    prediction = xgb.predict(input_df)[0]

    # SHAP for this prediction
    sv = explainer(input_df)
    shap_series = pd.Series(sv.values[0], index=feature_columns)
    top_features = shap_series.reindex(
        shap_series.abs().sort_values(ascending=False).index
    ).head(3)

    with output:
        print(f"Predicted Wait Time: {prediction:.1f} minutes")
        print("\nTop 3 drivers:")
        for feat, val in top_features.items():
            direction = "increased" if val > 0 else "reduced"
            print(f"  - {feat}: {val:.2f} ({direction} predicted wait time)")

predict_btn.on_click(predict)

display(
    widgets.VBox([
        widgets.HBox([urgency, region, season]),
        widgets.HBox([time_of_day, day_of_week]),
        widgets.HBox([nurse_ratio, specialist, facility]),
        widgets.HBox([visit_hour, visit_month, visit_day]),
        predict_btn,
        output
    ])
)

**Summary**

An interactive dashboard was developed to demonstrate the practical application of the trained model. An in-notebook interface using ipywidgets allows for quick experimentation within the development environment, enabling users to adjust input variables and observe changes in predicted wait times.

In addition, a Streamlit web application was implemented as a standalone prototype, providing a more accessible and user-friendly interface for non-technical users. The application allows users to input key patient and operational variables, generate real-time predictions, and view explainable insights through SHAP-based visualisations.

The dashboard highlights how predictive models can be integrated into decision-support tools. However, it currently requires manual input of operational variables such as staffing levels and bed capacity, which would typically be sourced automatically from hospital systems in a real-world deployment.

# **Conclusion**



This project successfully developed a machine learning system for predicting emergency department wait times using structured patient and operational data. The XGBoost model achieved strong predictive performance, meeting the predefined success criterion and outperforming baseline models.

Explainability was incorporated using SHAP, enabling both global and local interpretation of model predictions. This provides transparency and helps bridge the gap between predictive performance and practical usability in a healthcare context.

The development of interactive dashboards further demonstrated how the model could be applied in real-world scenarios. The Streamlit application, in particular, highlights the potential for integrating predictive models into user-facing systems.

However, several limitations remain. The use of a synthetic dataset restricts real-world validity, and some input variables may not be directly accessible to end users. Additionally, certain prediction behaviours observed during testing highlight the importance of careful validation.

Future work could focus on using real NHS data, implementing time-based validation strategies, and further refining the user interface to improve accessibility. Integrating the system with NHS data infrastructure would allow automatic retrieval of operational variables such as staffing levels and bed availability, removing the need for manual input and significantly improving usability for non-expert users.

Overall, the project demonstrates a complete pipeline from data analysis to deployment, combining predictive modelling with explainable AI to support decision-making in emergency healthcare environments.